In [48]:
import pandas as pd
import json
import numpy as np
import shutil
import os

In [49]:
dfs = []
for file in ['report_colab_2.csv']:#['report_file_store_colab.csv', 'report_file_store.csv', 'report_file_store_kaggle.csv']:
    df = pd.read_csv(file)
    df['source'] = file.removeprefix('report_file_store').removeprefix('_').removesuffix('.csv') or 'local'
    dfs.append(df)

In [50]:
report = pd.concat(dfs, ignore_index=True)

In [51]:
report = report[report['tags'].str.contains('testing')]

In [52]:
report = report[report['error'].isna() & ~report['result'].isna()]

In [53]:
report['result'] = [eval(x, {'np': np}) for x in report['result']]

In [54]:
report['trial_start'] = [json.loads(x)[0] for x in report['step_start']]
report['trial_start'] = pd.to_datetime(report['trial_start'])

In [55]:
report['trace'].apply(print)

{"val_score": [[18719.718469254673, {"fold": 0, "score": 18719.718469254673, "rmse": 13.881886493921437, "mae": 9.50509618900136, "r2": 0.8900805703402226, "map": 259767729.00768802, "cindex": 0.9165541790655138, "brier": 0.07442179303767286, "cdauc": 0.9897212281324759}], [18449.457577945876, {"fold": 1, "score": 18449.457577945876, "rmse": 13.943988134110407, "mae": 9.990926419755034, "r2": 0.8859984234823877, "map": 207610981.9373656, "cindex": 0.9193356894310394, "brier": 0.06515522289915641, "cdauc": 0.9794940496527512}], [18181.56378745836, {"fold": 2, "score": 18181.56378745836, "rmse": 14.20612691171224, "mae": 9.72094784306242, "r2": 0.8807947612185884, "map": 195791524.57665575, "cindex": 0.9259230353230251, "brier": 0.06954560952174317, "cdauc": 0.9790915806076084}], [18032.588302272234, {"fold": 3, "score": 18032.588302272234, "rmse": 14.294025551791236, "mae": 10.46980373887516, "r2": 0.8840913417323168, "map": 276659746.4871302, "cindex": 0.9191662339394019, "brier": 0.09

81      None
82      None
83      None
84      None
85      None
        ... 
3238    None
3239    None
3240    None
3241    None
3242    None
Name: trace, Length: 2912, dtype: object

In [56]:
metrics = ['score', 'rmse', 'r2', 'cindex']

In [57]:
report['train_score_std'] = [np.std([x[0] for x in json.loads(y).get('dt_score', [[np.nan]])]) for y in report['trace']]
for metric in metrics:
    for s in ['val', 'test']:
        report[s + '_' + metric] = [np.mean([x[1][metric] for x in json.loads(y).get(s+'_score', [[np.nan, {metric: np.nan}]])]) for y in report['trace']]
        report[s + '_' + metric + '_std'] = [np.std([x[1][metric] for x in json.loads(y).get(s+'_score', [[np.nan, {metric: np.nan}]])]) for y in report['trace']]

In [58]:
report.columns

Index(['uuid', 'optuna_trial_number', 'tags', 'step_names', 'step_start',
       'step_stop', 'result', 'error', 'prune_reason', 'vars', 'trace',
       'source', 'trial_start', 'train_score_std', 'val_score',
       'val_score_std', 'test_score', 'test_score_std', 'val_rmse',
       'val_rmse_std', 'test_rmse', 'test_rmse_std', 'val_r2', 'val_r2_std',
       'test_r2', 'test_r2_std', 'val_cindex', 'val_cindex_std', 'test_cindex',
       'test_cindex_std'],
      dtype='str')

In [59]:
report = report[(report['trial_start'] > pd.to_datetime('2026-10-12'))]

In [66]:
for arch in ['gru', 'thgnn', 'lstm', 'cnn']:
    for mode in ['e2e', 'supcon']:
        for ds in [f'FD00{x+1}' for x in range(4)]:
            cond = report['tags'].str.contains(arch) & report['tags'].str.contains(ds)
            if mode == 'e2e':
                cond &= ~report['tags'].str.contains('supcon')
            else:
                cond &= report['tags'].str.contains('supcon')
            sr = report[cond].sort_values('result').reset_index()
            print(f'{arch}\t{ds}\t{mode}', end='\t')
            for metric in metrics:
                for s in ['val', 'test']:
                    for kind in ['', '_std']:
                        print(f'{float(sr.loc[0, s+"_"+metric+kind])}'.replace('.', ','), end='\t')
            print()

gru	FD001	e2e	5132,96865234375	1803,0106264034337	462,0792236328125	0,0	9,676295471191406	1,4392623571179561	14,549256324768066	0,0	0,9439369559288024	0,017245870289853592	0,8681828379631042	0,0	0,9367529009128388	0,011875964299006642	0,8726337448559672	1,1102230246251565e-16	
gru	FD002	e2e	20597,36640625	3484,51431596543	673,9613647460938	0,0	12,438541793823243	0,9654133094827512	12,347755432128906	0,0	0,909151017665863	0,013482854875874896	0,9173073768615723	0,0	0,9158252200262151	0,006158730082534105	0,9064734544073911	1,1102230246251565e-16	
gru	FD003	e2e	6177,5494140625	2115,136130932593	335,9075012207031	0,0	9,567876434326172	1,0463915089091462	13,343076705932617	0,0	0,9465394973754883	0,011993710250669596	0,8839375972747803	0,0	0,9448285525504957	0,006796664560253586	0,895525494276795	0,0	
gru	FD004	e2e	33096,250390625	32332,98848944433	980,8740234375	0,0	11,283976554870605	1,5431468739612073	14,627685546875	0,0	0,9260308504104614	0,020621994190215487	0,8841769099235535	0,0	0,93

In [61]:
arch = 'gru'
ds = 'FD004'
r = report[
    ~report['tags'].str.contains('supcon')
    & report['tags'].str.contains(arch)
    & ~report['tags'].str.contains('margin_absolute')
    & report['tags'].str.contains(ds)].sort_values('result').reset_index()
x = r.loc[0, ['result', 'score_std', 'source', 'uuid']]
#r['vars'].apply(print)
d = {'colab': 'filestore_colab', 'kaggle': 'results-kaggle-9/filestore', 'local': 'adbiasminfilestore'}[x['source']]
x, d
file = '/mnt/sghdd/' + d + '/' + str(x['uuid']) + '.pth'
dest = '/home/sasha/Documents/analise-db-iasmin-2026/models/' + arch + '-e2e-' + ds.lower() + '.pth'
assert not os.path.exists(dest)
shutil.copy(file, dest)

KeyError: "['score_std'] not in index"